# Library

In [ ]:
import pandas as pd
import re
import os
import html
!pip install deep_translator
!pip install langdetect
from deep_translator import GoogleTranslator
from langdetect import detect, DetectorFactory
from tqdm import tqdm

# Loading Dataset

In [ ]:
DetectorFactory.seed = 0
file_name = 'data_komentar_mbg.csv'

if not os.path.exists(file_name):
    print(f"Error: File '{file_name}' tidak ditemukan!")
    exit()

df_raw = pd.read_csv(file_name, lineterminator='\n')
total_data_awal = len(df_raw)

df_raw.head()

Error: File 'data_komentar_mbg.csv' tidak ditemukan!


FileNotFoundError: [Errno 2] No such file or directory: 'data_komentar_mbg.csv'

# Dictionary

In [ ]:
url_kamus = 'https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv'
df_slang = pd.read_csv(url_kamus)
slang_dict = dict(zip(df_slang['slang'], df_slang['formal']))

kamus_tambahan = {
    "mbg": "makan bergizi gratis", "makan gratis": "makan bergizi gratis",
    "ps": "prabowo", "prab": "prabowo", "jkw": "jokowi",
    "ora": "tidak", "ra": "tidak", "nggih": "iya", "mboten": "tidak",
    "apik": "bagus", "elek": "jelek", "suwun": "terima kasih",
    "matur nuwun": "terima kasih", "piye": "bagaimana", "iki": "ini",
    "iku": "itu", "koe": "kamu", "kowe": "kamu", "aku": "saya",
    "wis": "sudah", "wes": "sudah", "durung": "belum", "urung": "belum",
    "tenan": "sekali", "banget": "sekali", "good": "bagus", "bad": "buruk",
    "yes": "iya", "no": "tidak", "fake": "palsu", "real": "asli",
    "hoax": "bohong", "bullshit": "omong kosong", "free": "gratis",
    "food": "makanan", "program": "program"
}
slang_dict.update(kamus_tambahan)

translator = GoogleTranslator(source='auto', target='id')

# Cleaning

In [ ]:
def basic_cleaning(text):
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = html.unescape(text)

    # Hapus sisa HTML entity dengan regex
    text = re.sub(r'&[a-zA-Z]{1,10};?', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    # Ubah ke huruf kecil semua
    text = text.lower()

    # Hapus Mojibake, Titik, Koma, dll.
    text = re.sub(r'[^a-z0-9\s!?]', ' ', text)

    # Hapus kata-kata sisa artifact HTML yang lolos
    artifact_words = {'quot', 'amp', 'nbsp', 'lt', 'gt', 'apos', 'hellip', 'ndash', 'mdash', 'laquo', 'raquo'}
    words = text.split()
    words = [w for w in words if w not in artifact_words]
    text = ' '.join(words)

    # Beri spasi pada tanda ! dan ?
    text = re.sub(r'([!?])', r' \1 ', text)

    # Hapus huruf berulang
    text = re.sub(r'([a-z])\1{2,}', r'\1', text)

    # Normalisasi kata slang
    words = text.split()
    normalized_words = [slang_dict.get(word, word) for word in words]
    text = ' '.join(normalized_words)

    # Hilangkan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # Rapatkan kembali tanda ! dan ?
    text = re.sub(r'\s+([!?])', r'\1', text)

    return text

print("Pembersihan dasar...")
tqdm.pandas(desc="Membersihkan Teks")
df_raw['cleaned_comment'] = df_raw['comment'].progress_apply(basic_cleaning)

In [ ]:
df_raw.head()

In [ ]:
df_raw.info()

# Filter

In [ ]:
df_bersih = df_raw.drop_duplicates(subset=['cleaned_comment']).reset_index(drop=True)
jumlah_duplikat_dihapus = len(df_raw) - len(df_bersih)

# Hapus < 5 Kata
def count_words(text):
    words_only = [w for w in text.split() if w.isalnum()]
    return len(words_only)

df_bersih['word_count'] = df_bersih['cleaned_comment'].apply(count_words)
df_final = df_bersih[df_bersih['word_count'] >= 5].reset_index(drop=True)
jumlah_pendek_dihapus = len(df_bersih) - len(df_final)

# Filter Keyword
keywords = [
    'makan', 'gizi', 'gratis', 'mbg', 'susu', 'siang',
    'anak', 'sekolah', 'siswa', 'anggaran', 'dana',
    'korupsi', 'pajak', 'menu', 'katering', 'janji', 'program'
]
keywords = [k.lower() for k in keywords]
pattern = r'\b(?:' + '|'.join(keywords) + r')\b'

df_filtered = df_final[df_final['cleaned_comment'].str.contains(pattern, regex=True, na=False)].reset_index(drop=True)
jumlah_tanpa_keyword_dihapus = len(df_final) - len(df_filtered)
df_final = df_filtered

In [ ]:
df_final.head()

In [ ]:
df_final.info()

# Save

In [ ]:
df_final['cleaned_comment'] = df_final['cleaned_comment'].apply(basic_cleaning)

df_final.insert(0, 'comment_id', range(1, len(df_final) + 1))
kolom_yang_disimpan = ['comment_id', 'cleaned_comment']
df_final = df_final[kolom_yang_disimpan]

total_data_dihapus = jumlah_duplikat_dihapus + jumlah_pendek_dihapus + jumlah_tanpa_keyword_dihapus
total_data_sisa = len(df_final)

print("\n" + "="*50)
print("             LAPORAN PEMBERSIHAN DATA")
print("="*50)
print(f"Total Data Mentah Awal    : {total_data_awal} baris")
print(f"(-) Komentar Duplikat     : {jumlah_duplikat_dihapus} baris dihapus")
print(f"(-) Komentar < 5 Kata     : {jumlah_pendek_dihapus} baris dihapus")
print(f"(-) Di Luar Topik/Keyword : {jumlah_tanpa_keyword_dihapus} baris dihapus")
print("-" * 50)
print(f"TOTAL DATA DIBUANG        : {total_data_dihapus} baris")
print(f"TOTAL DATA BERSIH (FINAL) : {total_data_sisa} baris")
print("="*50)

output_filename = 'data_komentar_mbg_bersih_translator.csv'
df_final.to_csv(output_filename, index=False)
print(f"\nSelesai! Data siap dilabeli oleh AI disimpan di '{output_filename}'")